# **🦜 How Fast Do You Forget? 13M Duolingo Sessions**
SESSION ~12,850,000 (February 28 to March 12, 2013)

This dataset tracks how human memory decays in the wild: 12,854,226 real vocabulary practice sessions from 115,222 Duolingo learners across 6 languages, compiled into analysis-ready forgetting curves, per-word difficulty tables, and course-level recall statistics. The primary source is the official replication data released by Duolingo for Settles & Meeder (2016), "A Trainable Spaced Repetition Model for Language Learning" (ACL) — compiled and structured by myself from the raw 379 MB trace file.

Average recall falls from 91.0% when a word was last practiced under an hour ago to 86.1% after 3+ months, declining monotonically across all nine lag bins — Ebbinghaus's forgetting curve, measured on real people at scale.

Methodology caveats: the traces come from a 13-day operational window in 2013, but the lags between practices (the variable the forgetting curve is built from) span minutes to many months. Recall does not rise cleanly with prior practice because harder words get practiced more — a built-in lesson in selection bias. p_recall is session-level, so short sessions produce coarse values (0, 0.5, 1).

## **Dataset Oerview**
Markdown documentation for each of the four files in your dataset based on the provided schemas

## 1. `forgetting_curve.csv`

This file tracks human memory decay metrics compiled into analysis-ready forgetting curves, aggregate binned tracking fields, and recall performance metrics.

| Column Name | Data Type / Format | Description |
| --- | --- | --- |
| `learning_language` | String (ISO Code) | The language being learned (e.g., `en`, `es`, `fr`, `de`, `it`, `pt`) or `all` for the combined language rollup. |
| `language_name` | String | Human-readable name of the language being learned (e.g., `English`, `German`, `All languages`). |
| `lag_bin` | String | Categorized time interval since the word was last practiced, binned from `<1 hour` up to `3+ months`. |
| `lag_bin_order` | Integer | Sorting order index for `lag_bin` (where `0` represents the shortest lag interval). |
| `practice_bin` | String | Binned representation of how many times the user had seen the word before this session (e.g., `1-2 exposures`, `3-4 exposures`, up to `20+ exposures`). |
| `practice_bin_order` | Integer | Sorting order index for `practice_bin` (where `0` represents the fewest exposures). |
| `n_traces` | Integer | Total number of practice sessions grouped within this specific metric cell. |
| `avg_session_recall` | Float | The mean per-session recall rate ($p\_recall$), ranging from `0` to `1`. |
| `item_recall_rate` | Float | Total number of correct answers divided by total exercises within this metric cell, ranging from `0` to `1`. |
| `pct_perfect_sessions` | Float | Percentage of learning sessions where every single exercise on the word was answered correctly. |


## 2. `language_courses.csv`

This file outlines aggregate metadata metrics broken down per active language course pairing.

| Column Name | Data Type / Format | Description |
| --- | --- | --- |
| `course` | String | The directional course pairing code mapping the user interface language to the target language (e.g., `es -> en`). |
| `ui_language` | String (ISO Code) | The user interface language of the speaker (e.g., `en`, `es`, `pt`). |
| `ui_language_name` | String | Human-readable name of the user interface language. |
| `learning_language` | String (ISO Code) | The target language being learned in the course. |
| `learning_language_name` | String | Human-readable name of the target language being learned. |
| `n_traces` | Integer | Total number of practice session events logged for this specific course. |
| `n_users` | Integer | Distinct number of users actively enrolled or taking this specific course. |
| `avg_session_recall` | Float | Average session recall rate across all sessions in this course pairing. |
| `item_recall_rate` | Float | Overall item-level correct answer rate for exercises within this course. |
| `avg_lag_days` | Float | The average number of days elapsed between word practices across the course. |
| `avg_prior_exposures` | Float | Average count of historical exposures users had to words prior to their sessions. |


## 3. `learning_traces_sample.csv`

This file contains the individual transaction-level session practice trace samples for words, serving as the core longitudinal user interaction dataset.

| Column Name | Data Type / Format | Description |
| --- | --- | --- |
| `practice_time` | Timestamp | Date and exact system timestamp when the practice session occurred. |
| `user_id` | String | Hashed unique identifier for the user. |
| `ui_language` | String (ISO Code) | The native interface language of the user. |
| `learning_language` | String (ISO Code) | The target language the user is practicing. |
| `surface_form` | String | The exact word string token as it appeared to the user in the exercise. |
| `lemma` | String | The base dictionary form (canonical root) of the surface word. |
| `pos` | String | Part of Speech tag assigned to the word token (e.g., `n`, `vblex`, `adj`). |
| `grammar_tags` | String | Morphological feature strings or grammatical attributes separated by semicolons (e.g., `pri;p3;sg`). |
| `lag_days` | Float | Time delta measured in days since the user last encountered/practiced this exact token. |
| `history_seen` | Integer | Total cumulative number of times the user has seen this specific word token historically. |
| `history_correct` | Integer | Total cumulative number of times the user has correctly answered this specific word token. |
| `session_seen` | Integer | The count of times this token was presented during the current practice session. |
| `session_correct` | Integer | The count of times the user got the token correct during the current practice session. |
| `p_recall` | Float | Calculated user performance probability metric for this session, spanning `0.0` to `1.0`. |
| `lexeme_id` | String | Unique programmatic identifier mapping to this specific word entity key. |


## 4. `word_difficulty.csv`

This file contains aggregated statistical metrics calculating difficulty ratings, distribution frequencies, and underlying tracking parameters isolated per unique lexeme/word entity.

| Column Name | Data Type / Format | Description |
| --- | --- | --- |
| `learning_language` | String (ISO Code) | The language category code that the token belongs to. |
| `language_name` | String | Full text name of the word's language. |
| `surface_form` | String | The text string representation of the word item. |
| `lemma` | String | The underlying root structural base of the word. |
| `pos` | String | Part of Speech tag categorization label. |
| `grammar_tags` | String | Semicolon-delimited grammatical attribute flags associated with the item profile. |
| `n_traces` | Integer | The total volume of observation trace sessions containing this specific lexeme. |
| `avg_session_recall` | Float | Average session-level recall score computed for this word across all encounters. |
| `item_recall_rate` | Float | Total aggregate performance rate (correct/total) calculated explicitly for this word. |
| `avg_prior_exposures` | Float | Mean number of prior user encounters historically logged for this word instance. |
| `difficulty_rank_in_course` | Integer | Relative sorted standard ranking denoting word complexity within its specific language pipeline structure. |
| `lexeme_id` | String | System master key indexing the unique word entity across the tables. |

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
from matplotlib import pyplot as plt

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kylefengkfeng209/how-fast-do-you-forget-13m-duolingo-sessions/language_courses.csv
/kaggle/input/datasets/kylefengkfeng209/how-fast-do-you-forget-13m-duolingo-sessions/forgetting_curve.csv
/kaggle/input/datasets/kylefengkfeng209/how-fast-do-you-forget-13m-duolingo-sessions/word_difficulty.csv
/kaggle/input/datasets/kylefengkfeng209/how-fast-do-you-forget-13m-duolingo-sessions/learning_traces_sample.csv


In [2]:
traces_df = pd.read_csv('/kaggle/input/datasets/kylefengkfeng209/how-fast-do-you-forget-13m-duolingo-sessions/learning_traces_sample.csv')
word_diff_df = pd.read_csv('/kaggle/input/datasets/kylefengkfeng209/how-fast-do-you-forget-13m-duolingo-sessions/word_difficulty.csv')
courses_df = pd.read_csv('/kaggle/input/datasets/kylefengkfeng209/how-fast-do-you-forget-13m-duolingo-sessions/language_courses.csv')
forgetting_df = pd.read_csv('/kaggle/input/datasets/kylefengkfeng209/how-fast-do-you-forget-13m-duolingo-sessions/forgetting_curve.csv')

# Convert lag from days to hours for localized interval mapping
lag_hours = traces_df['lag_days'] * 24.0

# Define the numerical cutoff boundaries for lag bins matching Duolingo's framework
# Bins: <1hr (0), 1-6hr (1), 7-24hr (2), 1-2 days (3), 3-7 days (4), 8-14 days (5), 15-30 days (6), 31-90 days (7), 3+ months (8)
lag_bins = [0, 1, 6, 24, 48, 168, 336, 720, 2160, np.inf]
traces_df['lag_bin_order'] = pd.cut(lag_hours, bins=lag_bins, labels=False, include_lowest=True)

# Define boundaries for prior historical exposures
# Bins: 1-2 (0), 3-4 (1), 5-9 (2), 10-19 (3), 20+ (4)
exposure_bins = [0, 2, 4, 9, 19, np.inf]
traces_df['practice_bin_order'] = pd.cut(traces_df['history_seen'], bins=exposure_bins, labels=False, include_lowest=True)

# Handle extreme/edge case scenarios gracefully by enforcing bounds
traces_df['lag_bin_order'] = traces_df['lag_bin_order'].fillna(0).astype(int)
traces_df['practice_bin_order'] = traces_df['practice_bin_order'].fillna(0).astype(int)

# A. Merge with Word Difficulty Table
# Dropping descriptive columns that already exist in traces to prevent collision suffixes (_x, _y)
word_subset = word_diff_df.columns.difference(['learning_language', 'language_name', 'surface_form', 'lemma', 'pos', 'grammar_tags'])
data = pd.merge(
    traces_df, 
    word_diff_df[word_subset.union(['lexeme_id'])], 
    on='lexeme_id', 
    how='left'
)

# B. Merge with Language Course Metadata Table
course_subset = courses_df.columns.difference(['ui_language_name', 'learning_language_name'])
data = pd.merge(
    data,
    courses_df[course_subset],
    on=['ui_language', 'learning_language'],
    how='left',
    suffixes=('', '_course_global')
)

# C. Merge with Forgetting Curve Table
forgetting_subset = forgetting_df.columns.difference(['language_name', 'lag_bin', 'practice_bin'])
data = pd.merge(
    data,
    forgetting_df[forgetting_subset],
    on=['learning_language', 'lag_bin_order', 'practice_bin_order'],
    how='left',
    suffixes=('', '_curve_baseline')
)

# Map original column structures to easy-to-understand definitions
rename_dictionary = {
    # Core User Session Attributes
    'practice_time':         'session_timestamp',
    'user_id':               'user_id',
    'ui_language':           'user_interface_lang_code',
    'learning_language':     'target_learning_lang_code',
    'p_recall':              'user_session_recall_probability',
    
    # Structural Word Details
    'lexeme_id':             'unique_word_key',
    'surface_form':          'word_string_raw',
    'lemma':                 'word_dictionary_root',
    'pos':                   'part_of_speech',
    'grammar_tags':          'grammatical_properties',
    
    # Session Level Longitudinal Counters
    'lag_days':              'days_since_last_practice',
    'history_seen':          'cumulative_user_exposures',
    'history_correct':       'cumulative_user_correct_answers',
    'session_seen':          'word_prompts_this_session',
    'session_correct':       'word_correct_this_session',
    
    # Feature Engineering Structural Indices
    'lag_bin_order':         'time_delay_bin_index',
    'practice_bin_order':    'prior_exposure_bin_index',
    
    # Word Difficulty Aggregates
    'n_traces':              'word_total_global_traces',
    'avg_session_recall':    'word_avg_global_recall',
    'item_recall_rate':      'word_total_global_accuracy',
    'avg_prior_exposures':   'word_avg_global_prior_exposures',
    'difficulty_rank_in_course': 'word_complexity_rank_in_course',
    
    # Course Macro Aggregates
    'n_traces_course_global': 'course_total_global_traces',
    'n_users':               'course_total_active_users',
    'avg_session_recall_course_global': 'course_avg_global_recall',
    'item_recall_rate_course_global':   'course_total_global_accuracy',
    'avg_lag_days':          'course_avg_delay_days',
    'avg_prior_exposures_course_global': 'course_avg_prior_exposures',
    
    # Memory Decay / Forgetting Curve Baseline Benchmarks
    'n_traces_curve_baseline':           'curve_bin_total_traces',
    'avg_session_recall_curve_baseline': 'curve_bin_expected_recall_baseline',
    'item_recall_rate_curve_baseline':   'curve_bin_expected_accuracy_baseline',
    'pct_perfect_sessions':              'curve_bin_pct_perfect_sessions'
}

data = data.rename(columns=rename_dictionary)

In [3]:
data.shape

(499436, 33)

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499436 entries, 0 to 499435
Data columns (total 33 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   session_timestamp                     499436 non-null  object 
 1   user_id                               499436 non-null  object 
 2   user_interface_lang_code              499436 non-null  object 
 3   target_learning_lang_code             499436 non-null  object 
 4   word_string_raw                       499436 non-null  object 
 5   word_dictionary_root                  499436 non-null  object 
 6   part_of_speech                        499436 non-null  object 
 7   grammatical_properties                448480 non-null  object 
 8   days_since_last_practice              499436 non-null  float64
 9   cumulative_user_exposures             499436 non-null  int64  
 10  cumulative_user_correct_answers       499436 non-null  int64  
 11  

### 1. The Core Event (What happened and who did it?)
*These columns tell you the "who, when, and what" of each individual language practice row.*

| Column Name | In Plain English | ML / Analysis Context |
| :--- | :--- | :--- |
| `session_timestamp` | The exact date and time the user practiced this word. | Useful for time-series splits or tracking daily user habits. |
| `user_id` | An anonymous, unique text code tracking a specific student. | Essential for grouping rows by user (e.g., computing a user's personal baseline learning speed). |
| `user_interface_lang_code` | The native language of the user's app interface (e.g., `en` for English speakers). | Helps control for the user's background or native language biases. |
| `target_learning_lang_code` | The language the user is actively trying to learn (e.g., `es` for Spanish). | Useful for filtering down your ML model to focus on a specific language track. |
| `course` | The full course pairing string (e.g., `es -> en`). | The unique combination of the user's interface language and their target language. |


### 2. Word Details (What exact item was practiced?)
*These columns describe the linguistic attributes of the word being studied.*

| Column Name | In Plain English | ML / Analysis Context |
| :--- | :--- | :--- |
| `word_string_raw` | The exact word as the user saw it on their screen (e.g., "amamos"). | Helpful for text features or identifying specific word lengths. |
| `word_dictionary_root` | The base dictionary form of the word (e.g., "amar"). | Groups different conjugations or plurals into a single root word. |
| `part_of_speech` | The grammatical role of the word (Noun, Verb, Adjective, etc.). | Verbs or adverbs might be fundamentally harder to remember than nouns. |
| `grammatical_properties` | Hidden linguistic details (like tense, plural, gender). | Can highlight specific rules that cause users to forget words more easily. |
| `unique_word_key` | Duolingo's internal ID code for this specific word item. | The direct primary key used to index this word globally across Duolingo's systems. |


### 3. User History Stats (How well does this specific person know this word?)
*These are your absolute goldmines for modeling the human forgetting curve—they map out this individual user's depth of exposure.*

| Column Name | In Plain English | ML / Analysis Context |
| :--- | :--- | :--- |
| `days_since_last_practice` | **The fundamental Forgetting Curve metric.** How many days (or fractions of a day) passed since this user last saw this word. | As this number grows, your target variable (`user_session_recall_probability`) should systematically drop. |
| `cumulative_user_exposures` | The total number of times this specific user has seen this word in the past. | Represents reinforcement. The higher this is, the flatter their forgetting curve should become. |
| `cumulative_user_correct_answers` | Out of those past exposures, how many times the user got it right. | Shows personal history. If this is much lower than total exposures, it's a "trouble word" for them. |
| `word_prompts_this_session` | How many times the word popped up inside *this single* study session. | Captures intra-session repetition or cramming effects. |
| `word_correct_this_session` | Out of those prompts *this session*, how many times they got it right. | Used directly to calculate your target recall variable. |
| `user_session_recall_probability` | **Your Main Target Variable ($y$).** The percentage score (0 to 1) of how well the user remembered the word right now. | This is what your ML model will ultimately try to predict based on time delays and history. |


### 4. Merge Keys & Global Baselines (What is normal for this word, course, or memory bin?)
*These columns provide the "macro" averages across the whole platform to help your model understand global defaults.*

| Column Name | In Plain English | ML / Analysis Context |
| :--- | :--- | :--- |
| `time_delay_bin_index` | An integer (0 to 8) categorizing the time delay into standard buckets (e.g., `<1 hour`, `1-2 days`). | The connecting bridge used to pull in the scientific memory decay curve parameters. |
| `prior_exposure_bin_index` | An integer (0 to 4) categorizing how many times an average person has seen the word before. | The second connecting bridge used to align rows with Duolingo's baseline memory groups. |
| `word_avg_global_prior_exposures` | On average across the app, how many times users see this specific word. | Tells you if this word generally appears early or late in the curriculum. |
| `word_avg_global_recall` | The default, app-wide baseline success rate for this exact word. | Gives your model a massive hint about the native difficulty of the word itself. |
| `difficulty_rank_in_language` | Where this word ranks in complexity relative to all other words in that course. | A highly engineered feature capturing relative item complexity. |
| `word_total_global_accuracy` | The global correct-to-incorrect ratio calculated for this word alone. | Another raw representation of how tough this item is globally. |
| `word_total_global_traces` | The absolute number of times this word shows up in the wider data pool. | Helps your model know if the global stats for this word are highly reliable or based on a small sample size. |
| `course_avg_delay_days` | The typical time gap standard for all users taking this specific course. | Captures normal course pacing (e.g., do Spanish learners practice more frequently than German learners?). |
| `course_avg_prior_exposures` | The macro-average number of times words are repeated inside this track. | Measures the structural repetition design built into the course track. |
| `course_avg_global_recall` | The total average memory success rate across every single student in this specific language course. | Establishes the macro difficulty of learning this target language track entirely. |
| `course_total_global_accuracy` | The absolute item-level correct answer rate across the entire course. | Similar to course recall, but focused directly on raw exercise performance. |
| `course_total_global_traces` | The total count of logged practice interactions within this specific language track. | Shows how popular the course track is in your global dataset. |
| `course_total_active_users` | The total number of distinct learners enrolled in this specific language pipeline. | Tells you how massive the sample population size is for this specific language course track. |
| `curve_bin_expected_recall_baseline` | **The scientific "Ebbinghaus" baseline.** What an average human's memory recall probability *should* be at this exact time delay and exposure level. | An exceptional feature for ML. Your model can use this as a starting baseline and adjust it up or down based on specific user history. |
| `curve_bin_expected_accuracy_baseline`| The default exercise accuracy expected for anyone landing inside this specific time/exposure bucket. | The exercise-level counterweight to the cognitive memory curve baseline. |
| `curve_bin_total_traces` | The size of the dataset pool used to calculate this specific forgetting curve baseline. | Confirms statistical weight for this time-lag segment. |
| `curve_bin_pct_perfect_sessions` | The percentage of sessions in this specific time/exposure tier where a user got every question flawlessly correct. | Highlights how frequently absolute mastery occurs at this stage of the decay curve. |